# ECG Signal Processing Workshop -- Notebook 2: ECG Preprocessing Pipeline

## Why Preprocessing Matters (Especially for Textile ECG)

Standard gelled electrodes (Ag/AgCl) provide excellent skin-electrode impedance and
therefore relatively clean signals. Textile electrodes -- like those in the **Baby Belt** --
use conductive fabric pressed against the skin without adhesive gel. This introduces
substantially more noise:

| Noise Source | Frequency Band | Origin |
|---|---|---|
| **Baseline wander** | 0.05 - 0.5 Hz | Respiration, body movement, electrode drift |
| **Powerline interference** | 50 / 60 Hz (+ harmonics) | Electromagnetic coupling from mains |
| **Motion artifact** | 1 - 10 Hz | Electrode-skin impedance changes during movement |
| **Muscle / contact noise** | > 40 Hz | EMG contamination, loose electrode contact |

In this notebook we walk through each preprocessing step, visualise its effect, and build
a **complete, reusable pipeline** that works for both BIOPAC (gelled, 1 kHz) and Baby Belt
(textile, 100 Hz) recordings.

### Notebook outline
1. Data loading (parsers copied inline from NB01 specification)
2. Noise characterisation on raw signals
3. Bandpass filtering (Butterworth, zero-phase)
4. Notch filtering (powerline removal)
5. Baseline wander removal (double median filter)
6. Synthetic noise addition & removal demo
7. Time-frequency analysis (Synchrosqueezed CWT & PyWavelets CWT)
8. Side-by-side BIOPAC vs Belt comparison

## 1. Environment Setup

The cell below imports every package used in this notebook.
Core scientific stack is required; `ssqueezepy` and `plotly` are optional
(gracefully skipped if missing).

In [ ]:
# =====================================================================
# USER CONFIGURATION
# =====================================================================
BIOPAC_FILE_PATH = r"sample_data/BIOPAC_04020062_9_Female_1st.txt"
BELT_FILE_PATH   = r"sample_data/BABY_BELT_04020062_9_Female_1st.csv"
OUTPUT_DIR       = r"outputs/NB02"
# =====================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.signal import butter, sosfiltfilt, iirnotch, tf2sos, medfilt, welch
from io import StringIO
import os, warnings

HAS_PLOTLY = False
try:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots as plotly_subplots
    HAS_PLOTLY = True
except ImportError:
    print("plotly not installed -- interactive plots will use matplotlib fallback.")

HAS_NK2 = False
try:
    import neurokit2 as nk
    HAS_NK2 = True
except ImportError:
    print("neurokit2 not installed.")

HAS_PYW = False
try:
    import pywt
    HAS_PYW = True
except ImportError:
    print("PyWavelets (pywt) not installed.")

HAS_SSQ = False
try:
    from ssqueezepy import ssq_cwt, issq_cwt
    HAS_SSQ = True
except ImportError:
    print("ssqueezepy not installed. SSQ demo will be skipped.")

from tqdm.notebook import tqdm

warnings.filterwarnings("ignore")
np.random.seed(42)

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("seaborn-whitegrid")

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory ready: {os.path.abspath(OUTPUT_DIR)}")

## 2. Data Loading

We define `parse_biopac()` and `parse_belt()` inline (identical to NB01) so this
notebook is self-contained.

**BIOPAC format** -- AcqKnowledge `.txt` export: metadata header followed by
tab-separated columns (`milliSec`, `CH1` ... `CH47`). Sampling rate is encoded
in the header as `N msec/sample`. The raw ECG lives in **CH2** (ECG module output
**CH1** is sync/auxiliary (not the ECG trace). **CH40** is the ECG (mV), hardware-filtered 0.05–100 Hz AHA — used as `ecg_raw` for the software pipeline below.

**Baby Belt format** -- streamed CSV: comma-separated rows with `PC_Time`
(elapsed seconds), `ECG` (raw ADC counts), plus accelerometer, gyroscope, SpO2,
etc. Nominal rate is approx 100 Hz but inter-arrival times may jitter due to BLE scheduling.

In [ ]:
def parse_biopac(filepath):
    """Parse BIOPAC AcqKnowledge exported text file.

    Reads the tab-separated export with metadata header, channel
    descriptions, and numeric data rows.

    Parameters
    ----------
    filepath : str
        Path to the BIOPAC .txt export file.

    Returns
    -------
    dict
        ecg_raw      : BIOPAC ECG (mV), channel CH40 (CH1 is sync — not ECG)
        ecg_filtered : BIOPAC AHA-filtered ECG (mV), channel CH40
        time_s       : time vector in seconds
        fs           : sampling frequency in Hz
        metadata     : recording metadata dict
    """
    with open(filepath, "r", encoding="utf-8") as f:
        lines = f.readlines()

    metadata = {}
    sample_rate_ms = None
    header_idx = None

    for idx, line in enumerate(lines):
        stripped = line.strip()
        if not stripped:
            continue
        if "msec/sample" in stripped and sample_rate_ms is None:
            sample_rate_ms = float(stripped.split()[0])
        if stripped.startswith("Recording on:"):
            metadata["recording_time"] = stripped.split(": ", 1)[1]
        if stripped.startswith("milliSec"):
            header_idx = idx
            break

    if sample_rate_ms is None:
        raise ValueError("Could not find sample rate (msec/sample) in header.")
    if header_idx is None:
        raise ValueError("Could not find data header line starting with milliSec.")

    fs = 1000.0 / sample_rate_ms

    col_names = [c.strip() for c in lines[header_idx].strip().split("\t") if c.strip()]

    data_text = "".join(lines[header_idx + 2:])
    df = pd.read_csv(StringIO(data_text), sep="\t", header=None, on_bad_lines="skip")
    df = df.dropna(how="all", axis=1)
    df.columns = col_names[:len(df.columns)]

    time_s = df["milliSec"].values / 1000.0
    ecg_raw = df["CH40"].values.astype(float)
    ecg_filtered = df["CH40"].values.astype(float) if "CH40" in df.columns else ecg_raw.copy()

    duration = len(ecg_raw) / fs
    print(f"  BIOPAC: {len(ecg_raw):,} samples | {fs:.0f} Hz | {duration:.1f} s")
    print(f"  Raw ECG range: [{ecg_raw.min():.4f}, {ecg_raw.max():.4f}] mV")

    return dict(ecg_raw=ecg_raw, ecg_filtered=ecg_filtered, time_s=time_s,
                fs=fs, metadata=metadata)


def parse_belt(filepath):
    """Parse Baby Belt CSV file.

    Reads the comma-separated sensor CSV with columns including
    PC_Time, ECG, accelerometer, gyroscope, SpO2, etc.

    Parameters
    ----------
    filepath : str
        Path to the Baby Belt .csv file.

    Returns
    -------
    dict
        ecg      : raw ECG signal (ADC counts)
        time_s   : elapsed time in seconds
        fs       : estimated sampling frequency in Hz
        df       : full pandas DataFrame
        metadata : recording metadata dict
    """
    df = pd.read_csv(filepath)

    time_s = df["PC_Time"].values.astype(float)
    ecg = df["ECG"].values.astype(float)

    dt_median = np.median(np.diff(time_s))
    fs = round(1.0 / dt_median) if dt_median > 0 else 100.0

    duration = time_s[-1] - time_s[0]
    print(f"  Belt:   {len(ecg):,} samples | ~{fs:.0f} Hz | {duration:.1f} s")
    print(f"  Raw ECG range: [{ecg.min():.0f}, {ecg.max():.0f}] (ADC counts)")

    sensor_cols = [c for c in df.columns
                   if c not in ["PC_Time", "Seq", "Tx_ms", "Rx_ms", "Latency", "InterArrival"]]
    return dict(ecg=ecg, time_s=time_s, fs=fs, df=df,
                metadata=dict(estimated_fs=fs, n_samples=len(ecg),
                              duration_s=duration, available_signals=sensor_cols))


# --- Load both datasets ---
print("Loading BIOPAC data...")
biopac = parse_biopac(BIOPAC_FILE_PATH)

print("\nLoading Baby Belt data...")
belt = parse_belt(BELT_FILE_PATH)

## 3. Noise Characterisation

Before filtering, it is instructive to look at the **raw** signals and understand
the spectral content of each noise source.

### The four main ECG contaminants

1. **Baseline Wander (0.05 - 0.5 Hz)** -- slow undulation caused by respiration
   and postural changes. Particularly severe in textile electrodes that lack
   adhesive coupling.
2. **Powerline Interference (50 / 60 Hz + harmonics)** -- narrow-band sinusoidal
   contamination from electromagnetic coupling with the mains supply.
3. **Motion Artifact (1 - 10 Hz)** -- abrupt, irregular deflections caused by
   electrode-skin impedance variations during movement. Overlaps the QRS
   frequency band, making it the hardest to remove without distorting morphology.
4. **Muscle / Contact Noise (> 40 Hz)** -- broadband high-frequency noise from
   skeletal muscle (EMG) activity and poor electrode contact.

The plot below shows a 10-second segment of the Belt raw ECG (textile electrode)
with approximate noise regions annotated. Compare this with the BIOPAC reference --
the difference in baseline stability is immediately apparent.

In [ ]:
VIEW_START_S = 5.0
VIEW_END_S   = 15.0


def get_window(signal, time_s, t_start, t_end):
    """Extract a time window from a signal array.

    Parameters
    ----------
    signal : np.ndarray
        Full signal array.
    time_s : np.ndarray
        Corresponding time vector in seconds.
    t_start : float
        Window start in seconds.
    t_end : float
        Window end in seconds.

    Returns
    -------
    tuple of (np.ndarray, np.ndarray)
        (windowed_signal, windowed_time)
    """
    mask = (time_s >= t_start) & (time_s < t_end)
    return signal[mask], time_s[mask]


bp_ecg_win, bp_t_win = get_window(biopac["ecg_raw"], biopac["time_s"], VIEW_START_S, VIEW_END_S)
bl_ecg_win, bl_t_win = get_window(belt["ecg"], belt["time_s"], VIEW_START_S, VIEW_END_S)

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

axes[0].plot(bp_t_win, bp_ecg_win, color="steelblue", linewidth=0.6)
axes[0].set_title("BIOPAC -- Raw ECG (gelled electrode, 1 kHz)", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Amplitude (mV)")

axes[1].plot(bl_t_win, bl_ecg_win, color="darkorange", linewidth=0.6)
axes[1].set_title("Baby Belt -- Raw ECG (textile electrode, ~100 Hz)", fontsize=13, fontweight="bold")
axes[1].set_ylabel("Amplitude (ADC counts)")
axes[1].set_xlabel("Time (s)")

noise_colors = {"Baseline\nwander": "#2ca02c", "Motion\nartifact": "#d62728",
                "Muscle/\ncontact": "#9467bd", "Powerline": "#ff7f0e"}
legend_patches = [mpatches.Patch(color=c, label=l, alpha=0.3) for l, c in noise_colors.items()]
axes[1].legend(handles=legend_patches, loc="upper right", fontsize=9, title="Noise sources")

for ax in axes:
    ax.grid(True, alpha=0.3)

fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "raw_ecg_comparison.png"), dpi=150, bbox_inches="tight")
plt.show()

## 4. Bandpass Filtering

A **Butterworth bandpass filter** is the workhorse of ECG preprocessing. We
implement it with second-order sections (`sos`) for numerical stability and
apply it with `sosfiltfilt` for **zero-phase** operation (no group delay distortion).

### Cutoff frequency choices

| Application | Low cutoff | High cutoff | Rationale |
|---|---|---|---|
| **QRS-focused** | 0.5 Hz | 40 Hz | Standard for R-peak detection -- removes baseline wander and HF noise while preserving QRS morphology |
| **Clinical / diagnostic** | 0.05 Hz | 100 Hz (or 150 Hz) | Preserves ST segment and P/T wave detail for clinical interpretation |

We demonstrate both bands below.

In [ ]:
def bandpass_filter(signal, fs, lowcut=0.5, highcut=40.0, order=5):
    """Apply zero-phase Butterworth bandpass filter.

    Parameters
    ----------
    signal : np.ndarray
        Input signal.
    fs : float
        Sampling frequency in Hz.
    lowcut : float
        Lower cutoff frequency in Hz.
    highcut : float
        Upper cutoff frequency in Hz.
    order : int
        Filter order.

    Returns
    -------
    np.ndarray
        Filtered signal.
    """
    nyq = fs / 2.0
    hi = min(highcut, nyq - 1.0)
    sos = butter(order, [lowcut, hi], btype="band", fs=fs, output="sos")
    return sosfiltfilt(sos, signal)


# Apply QRS-focused bandpass (0.5-40 Hz) to both signals
bp_ecg_bp = bandpass_filter(biopac["ecg_raw"], biopac["fs"], 0.5, 40.0)
bl_ecg_bp = bandpass_filter(belt["ecg"], belt["fs"], 0.5, 40.0)

# Apply wider clinical bandpass -- only meaningful for BIOPAC at 1 kHz
bp_ecg_wide = bandpass_filter(biopac["ecg_raw"], biopac["fs"], 0.5, 100.0)
bl_ecg_wide = bandpass_filter(belt["ecg"], belt["fs"], 0.5, min(48.0, belt["fs"] / 2 - 1))

# --- 2x2 subplot: raw vs bandpass for each device ---
fig, axes = plt.subplots(2, 2, figsize=(16, 8))

titles = [
    ("BIOPAC -- Raw ECG", "BIOPAC -- Bandpass 0.5-40 Hz"),
    ("Belt -- Raw ECG", "Belt -- Bandpass 0.5-40 Hz"),
]
signals_raw = [biopac["ecg_raw"], belt["ecg"]]
signals_filt = [bp_ecg_bp, bl_ecg_bp]
times = [biopac["time_s"], belt["time_s"]]
colors_raw = ["steelblue", "darkorange"]
colors_filt = ["navy", "darkred"]

for row in range(2):
    t = times[row]
    mask = (t >= VIEW_START_S) & (t < VIEW_END_S)
    tw = t[mask]

    axes[row, 0].plot(tw, signals_raw[row][mask], color=colors_raw[row], linewidth=0.5, label="Raw")
    axes[row, 0].set_title(titles[row][0], fontweight="bold")
    axes[row, 0].set_ylabel("Amplitude")
    axes[row, 0].legend(loc="upper right")
    axes[row, 0].grid(True, alpha=0.3)

    axes[row, 1].plot(tw, signals_filt[row][mask], color=colors_filt[row], linewidth=0.5, label="Filtered")
    axes[row, 1].set_title(titles[row][1], fontweight="bold")
    axes[row, 1].set_ylabel("Amplitude")
    axes[row, 1].legend(loc="upper right")
    axes[row, 1].grid(True, alpha=0.3)

for ax in axes[1, :]:
    ax.set_xlabel("Time (s)")

fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "bandpass_comparison_2x2.png"), dpi=150, bbox_inches="tight")
plt.show()

print("Bandpass filtering complete.")

## 5. Notch Filtering (Powerline Removal)

Powerline interference (50 Hz in most of the world, 60 Hz in the Americas)
appears as a narrow spectral peak. An **IIR notch filter** removes it with
minimal distortion of surrounding frequencies.

The **quality factor** `Q` controls the notch width: higher Q = narrower notch.
A value of 30 is standard for ECG; it removes the target frequency without
significantly affecting the QRS complex.

Below we apply a 60 Hz notch and compare the **power spectral density (PSD)**
before and after, zoomed to the 50-70 Hz range.

In [ ]:
def notch_filter(signal, fs, freq=60.0, quality=30.0):
    """Apply IIR notch filter at specified frequency.

    Parameters
    ----------
    signal : np.ndarray
        Input signal.
    fs : float
        Sampling frequency in Hz.
    freq : float
        Notch frequency in Hz.
    quality : float
        Quality factor (higher = narrower notch).

    Returns
    -------
    np.ndarray
        Filtered signal.
    """
    nyquist = fs / 2.0
    if freq >= nyquist:
        print(f'  [notch_filter] Skipping {freq:.0f} Hz notch — '
              f'above Nyquist ({nyquist:.0f} Hz) for fs={fs:.0f} Hz')
        return signal
    b, a = iirnotch(freq, quality, fs)
    sos = tf2sos(b, a)
    return sosfiltfilt(sos, signal)


# Apply notch at 60 Hz to BIOPAC (high enough fs to resolve it)
bp_ecg_notched = notch_filter(biopac["ecg_raw"], biopac["fs"], freq=60.0, quality=30.0)

# PSD comparison (before and after notch)
nperseg_bp = min(4096, len(biopac["ecg_raw"]))
f_before, psd_before = welch(biopac["ecg_raw"], fs=biopac["fs"], nperseg=nperseg_bp)
f_after, psd_after   = welch(bp_ecg_notched, fs=biopac["fs"], nperseg=nperseg_bp)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Full PSD
axes[0].semilogy(f_before, psd_before, color="steelblue", alpha=0.7, label="Before notch")
axes[0].semilogy(f_after, psd_after, color="crimson", alpha=0.7, label="After 60 Hz notch")
axes[0].set_title("BIOPAC -- Power Spectral Density (full range)", fontweight="bold")
axes[0].set_xlabel("Frequency (Hz)")
axes[0].set_ylabel("PSD (V^2/Hz)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_xlim(0, 200)

# Zoomed to 50-70 Hz
mask_zoom = (f_before >= 50) & (f_before <= 70)
axes[1].plot(f_before[mask_zoom], psd_before[mask_zoom], "o-", color="steelblue",
             label="Before notch", markersize=3)
axes[1].plot(f_after[mask_zoom], psd_after[mask_zoom], "o-", color="crimson",
             label="After 60 Hz notch", markersize=3)
axes[1].axvline(60, color="gray", linestyle="--", alpha=0.5, label="60 Hz")
axes[1].set_title("PSD Zoomed: 50-70 Hz", fontweight="bold")
axes[1].set_xlabel("Frequency (Hz)")
axes[1].set_ylabel("PSD (V^2/Hz)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "notch_filter_psd.png"), dpi=150, bbox_inches="tight")
plt.show()

## 6. Baseline Wander Removal

Baseline wander is the slow drift (< 0.5 Hz) superimposed on the ECG.
A popular non-linear approach is the **double median filter**:

1. Apply a median filter with a window of approx 200 ms (captures the QRS width)
2. Apply a second median filter with a window of approx 600 ms (captures the full beat width)
3. Subtract the estimated baseline from the original signal

This preserves sharp QRS morphology better than a simple high-pass filter,
especially at aggressive cutoff frequencies.

In [ ]:
def remove_baseline(signal, fs):
    """Remove baseline wander using double median filter.

    Parameters
    ----------
    signal : np.ndarray
        Input signal.
    fs : float
        Sampling frequency in Hz.

    Returns
    -------
    tuple of (np.ndarray, np.ndarray)
        (signal_corrected, estimated_baseline)
    """
    win1 = int(0.2 * fs) | 1  # ~200 ms, ensure odd
    win2 = int(0.6 * fs) | 1  # ~600 ms, ensure odd
    baseline = medfilt(medfilt(signal, win1), win2)
    return signal - baseline, baseline


# Full preprocessing chain for BIOPAC
bp_step1 = notch_filter(biopac["ecg_raw"], biopac["fs"], freq=60.0)
bp_step2 = bandpass_filter(bp_step1, biopac["fs"], 0.5, 40.0)
bp_step3, bp_baseline = remove_baseline(bp_step2, biopac["fs"])

# Full preprocessing chain for Belt
bl_step1 = bandpass_filter(belt["ecg"], belt["fs"], 0.5, 40.0)
bl_step2, bl_baseline = remove_baseline(bl_step1, belt["fs"])

# --- 4-panel plot: BIOPAC pipeline stages ---
t_bp = biopac["time_s"]
mask_bp = (t_bp >= VIEW_START_S) & (t_bp < VIEW_END_S)

fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)

axes[0].plot(t_bp[mask_bp], biopac["ecg_raw"][mask_bp], color="steelblue", linewidth=0.5)
axes[0].set_title("Stage 0 -- Raw ECG (BIOPAC)", fontweight="bold")
axes[0].set_ylabel("mV")

axes[1].plot(t_bp[mask_bp], bp_step2[mask_bp], color="teal", linewidth=0.5)
axes[1].set_title("Stage 1 -- After Notch (60 Hz) + Bandpass (0.5-40 Hz)", fontweight="bold")
axes[1].set_ylabel("mV")

axes[2].plot(t_bp[mask_bp], bp_step2[mask_bp], color="gray", linewidth=0.4, alpha=0.5,
             label="Before baseline removal")
axes[2].plot(t_bp[mask_bp], bp_baseline[mask_bp], color="red", linewidth=1.5,
             label="Estimated baseline")
axes[2].set_title("Stage 2a -- Estimated Baseline (double median filter)", fontweight="bold")
axes[2].set_ylabel("mV")
axes[2].legend(loc="upper right")

axes[3].plot(t_bp[mask_bp], bp_step3[mask_bp], color="darkgreen", linewidth=0.5)
axes[3].set_title("Stage 2b -- After Baseline Removal (final)", fontweight="bold")
axes[3].set_ylabel("mV")
axes[3].set_xlabel("Time (s)")

for ax in axes:
    ax.grid(True, alpha=0.3)

fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "baseline_removal_stages.png"), dpi=150, bbox_inches="tight")
plt.show()

## 7. Synthetic Noise Addition (Exploration)

To develop intuition about how each noise type affects the ECG and how well
our pipeline removes it, we can **add synthetic noise** to a clean segment
and then observe the cleanup.

We implement four synthetic noise generators, loosely based on the augmentation
pipeline used for training RPNet (see `reference_codes/ECG_augumentor_dataset.py`):

| Generator | What it produces | Key parameter |
|---|---|---|
| `add_synthetic_baseline_wander` | Sum of low-frequency sinusoids (0.01-0.5 Hz) | `amplitude_fraction` |
| `add_synthetic_gaussian_noise` | White Gaussian noise at a target SNR | `snr_db` |
| `add_synthetic_powerline` | 60 Hz sinusoid | `amplitude_fraction` |
| `add_synthetic_emg_noise` | Band-limited (10-48 Hz) coloured noise | `snr_db` |

**Adjust the configuration block** in the cell after next to explore different intensities.

In [ ]:
def add_synthetic_baseline_wander(signal, fs, n_components=4, max_freq=0.5,
                                  amplitude_fraction=0.3, seed=42):
    """Add synthetic baseline wander to signal.

    Parameters
    ----------
    signal : np.ndarray
        Input signal.
    fs : float
        Sampling frequency in Hz.
    n_components : int
        Number of sinusoidal components.
    max_freq : float
        Maximum wander frequency in Hz.
    amplitude_fraction : float
        Wander amplitude as fraction of signal peak-to-peak.
    seed : int
        Random seed for reproducibility.

    Returns
    -------
    tuple of (np.ndarray, np.ndarray)
        (noisy_signal, wander_component)
    """
    rng = np.random.default_rng(seed)
    t = np.arange(len(signal)) / fs
    wander = np.zeros_like(signal, dtype=float)
    sig_amp = np.ptp(signal) + 1e-9
    for _ in range(n_components):
        freq = rng.uniform(0.01, max_freq)
        amp = rng.uniform(0.1, amplitude_fraction) * sig_amp
        phase = rng.uniform(0, 2 * np.pi)
        wander += amp * np.sin(2 * np.pi * freq * t + phase)
    return signal + wander, wander


def add_synthetic_gaussian_noise(signal, snr_db=20, seed=42):
    """Add Gaussian noise at specified SNR.

    Parameters
    ----------
    signal : np.ndarray
        Input signal.
    snr_db : float
        Target signal-to-noise ratio in dB.
    seed : int
        Random seed for reproducibility.

    Returns
    -------
    tuple of (np.ndarray, np.ndarray)
        (noisy_signal, noise_component)
    """
    rng = np.random.default_rng(seed)
    sig_power = np.mean(signal ** 2) + 1e-12
    noise_std = np.sqrt(sig_power / (10 ** (snr_db / 10)))
    noise = rng.normal(0, noise_std, len(signal))
    return signal + noise, noise


def add_synthetic_powerline(signal, fs, freq=60.0, amplitude_fraction=0.1, seed=42):
    """Add synthetic powerline interference.

    Parameters
    ----------
    signal : np.ndarray
        Input signal.
    fs : float
        Sampling frequency in Hz.
    freq : float
        Powerline frequency in Hz.
    amplitude_fraction : float
        PLI amplitude as fraction of signal peak-to-peak.
    seed : int
        Random seed for reproducibility.

    Returns
    -------
    tuple of (np.ndarray, np.ndarray)
        (noisy_signal, pli_component)
    """
    rng = np.random.default_rng(seed)
    t = np.arange(len(signal)) / fs
    amp = amplitude_fraction * (np.ptp(signal) + 1e-9)
    pli = amp * np.sin(2 * np.pi * freq * t + rng.uniform(0, 2 * np.pi))
    return signal + pli, pli


def add_synthetic_emg_noise(signal, fs, snr_db=15, seed=42):
    """Add synthetic EMG (muscle) noise.

    Band-limited noise in the 10-48 Hz range scaled to the
    requested SNR relative to the input signal.

    Parameters
    ----------
    signal : np.ndarray
        Input signal.
    fs : float
        Sampling frequency in Hz.
    snr_db : float
        Target signal-to-noise ratio in dB.
    seed : int
        Random seed for reproducibility.

    Returns
    -------
    tuple of (np.ndarray, np.ndarray)
        (noisy_signal, emg_component)
    """
    rng = np.random.default_rng(seed)
    raw_noise = rng.standard_normal(len(signal))
    hi = min(48.0, fs / 2 - 1)
    sos = butter(5, [10.0, hi], btype="bandpass", fs=fs, output="sos")
    emg = sosfiltfilt(sos, raw_noise)
    sig_rms = np.sqrt(np.mean(signal ** 2)) + 1e-9
    noise_rms = sig_rms / (10 ** (snr_db / 20))
    emg = emg / (np.std(emg) + 1e-9) * noise_rms
    return signal + emg, emg


print("Synthetic noise functions defined.")

In [ ]:
# =====================================================================
# NOISE CONFIGURATION -- Adjust these parameters to explore
# =====================================================================
BW_N_COMPONENTS = 4
BW_MAX_FREQ     = 0.5
BW_AMPLITUDE    = 0.3
GAUSS_SNR_DB    = 20
PLI_FREQ        = 60.0
PLI_AMPLITUDE   = 0.1
EMG_SNR_DB      = 15
# =====================================================================

# Use a clean 10-second segment from BIOPAC (already preprocessed)
seg_start = int(VIEW_START_S * biopac["fs"])
seg_end   = int(VIEW_END_S * biopac["fs"])
clean_seg = bp_step3[seg_start:seg_end].copy()
t_seg = biopac["time_s"][seg_start:seg_end]
fs_seg = biopac["fs"]

# Generate each noise type
noisy_bw, noise_bw     = add_synthetic_baseline_wander(clean_seg, fs_seg, BW_N_COMPONENTS,
                                                        BW_MAX_FREQ, BW_AMPLITUDE)
noisy_gauss, noise_g   = add_synthetic_gaussian_noise(clean_seg, GAUSS_SNR_DB)
noisy_pli, noise_pli   = add_synthetic_powerline(clean_seg, fs_seg, PLI_FREQ, PLI_AMPLITUDE)
noisy_emg, noise_emg   = add_synthetic_emg_noise(clean_seg, fs_seg, EMG_SNR_DB)

# --- 5-panel figure ---
fig, axes = plt.subplots(5, 1, figsize=(14, 14), sharex=True)

panels = [
    ("Clean ECG (reference)", clean_seg, "darkgreen"),
    (f"+ Baseline Wander ({BW_N_COMPONENTS} components, amp={BW_AMPLITUDE})", noisy_bw, "#2ca02c"),
    (f"+ Gaussian Noise (SNR={GAUSS_SNR_DB} dB)", noisy_gauss, "#1f77b4"),
    (f"+ Powerline Interference ({PLI_FREQ} Hz, amp={PLI_AMPLITUDE})", noisy_pli, "#ff7f0e"),
    (f"+ EMG Noise (SNR={EMG_SNR_DB} dB)", noisy_emg, "#d62728"),
]

for ax, (title, sig, color) in zip(axes, panels):
    ax.plot(t_seg, sig, color=color, linewidth=0.5)
    ax.set_title(title, fontweight="bold", fontsize=11)
    ax.set_ylabel("Amplitude")
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Time (s)")

fig.suptitle("Synthetic Noise Types Applied to Clean ECG Segment",
             fontsize=14, fontweight="bold", y=1.01)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "synthetic_noise_types.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Combine ALL noise types onto the clean segment
combined_noisy = clean_seg.copy()
combined_noisy, _ = add_synthetic_baseline_wander(combined_noisy, fs_seg,
                                                   BW_N_COMPONENTS, BW_MAX_FREQ, BW_AMPLITUDE, seed=42)
combined_noisy, _ = add_synthetic_gaussian_noise(combined_noisy, GAUSS_SNR_DB, seed=43)
combined_noisy, _ = add_synthetic_powerline(combined_noisy, fs_seg, PLI_FREQ, PLI_AMPLITUDE, seed=44)
combined_noisy, _ = add_synthetic_emg_noise(combined_noisy, fs_seg, EMG_SNR_DB, seed=45)

# Clean it up with our pipeline
step_a = notch_filter(combined_noisy, fs_seg, freq=PLI_FREQ)
step_b = bandpass_filter(step_a, fs_seg, 0.5, 40.0)
step_c, _ = remove_baseline(step_b, fs_seg)

# --- 4-panel comparison ---
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)

axes[0].plot(t_seg, clean_seg, color="darkgreen", linewidth=0.5)
axes[0].set_title("Original Clean Segment", fontweight="bold")

axes[1].plot(t_seg, combined_noisy, color="red", linewidth=0.5)
axes[1].set_title("All Noise Types Combined", fontweight="bold")

axes[2].plot(t_seg, step_b, color="teal", linewidth=0.5)
axes[2].set_title("After Notch + Bandpass", fontweight="bold")

axes[3].plot(t_seg, step_c, color="navy", linewidth=0.5)
axes[3].plot(t_seg, clean_seg, color="darkgreen", linewidth=0.5, alpha=0.4, label="Original clean")
axes[3].set_title("After Full Pipeline (+ original overlay)", fontweight="bold")
axes[3].legend(loc="upper right")

for ax in axes:
    ax.set_ylabel("Amplitude")
    ax.grid(True, alpha=0.3)
axes[-1].set_xlabel("Time (s)")

# Residual error
residual = step_c - clean_seg
rmse = np.sqrt(np.mean(residual ** 2))
corr = np.corrcoef(clean_seg, step_c)[0, 1]
axes[3].text(0.01, 0.05, f"RMSE={rmse:.4f}  r={corr:.4f}",
             transform=axes[3].transAxes, fontsize=10,
             bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8))

fig.suptitle("Pipeline Noise Removal Demo", fontsize=14, fontweight="bold", y=1.01)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "pipeline_noise_removal_demo.png"), dpi=150, bbox_inches="tight")
plt.show()

print(f"Reconstruction quality -- RMSE: {rmse:.6f}, Correlation: {corr:.6f}")

## 8. Synchrosqueezed CWT (ssqueezepy)

The **Synchrosqueezed Continuous Wavelet Transform (SSQ-CWT)** improves on
the standard CWT by "reassigning" energy in the time-frequency plane to
sharpen frequency localisation. This gives:

- **Better frequency resolution** than standard CWT at the same scale
- **Invertibility** -- we can reconstruct the signal from a selected frequency band
- Cleaner separation of overlapping components (e.g., QRS from baseline)

The trade-off is higher computational cost. We demonstrate it on a **10-second
segment** for practicality.

> **Note:** `ssqueezepy` is optional. If not installed, this section is skipped gracefully.

In [ ]:
def ssq_ecg_band_reconstruct(signal, fs, fmin=5, fmax=40):
    """Reconstruct ECG from synchrosqueezed CWT within frequency band.

    Parameters
    ----------
    signal : np.ndarray
        Input signal.
    fs : float
        Sampling frequency in Hz.
    fmin : float
        Minimum frequency for reconstruction.
    fmax : float
        Maximum frequency for reconstruction.

    Returns
    -------
    np.ndarray
        Reconstructed signal from selected frequency band.
    """
    if not HAS_SSQ:
        print("ssqueezepy not available -- returning input unchanged.")
        return signal
    Tx, Wx, ssq_freqs, scales, *_ = ssq_cwt(signal, fs=fs)
    band_mask = (ssq_freqs >= fmin) & (ssq_freqs <= fmax)
    Tx_band = np.zeros_like(Tx)
    Tx_band[band_mask, :] = Tx[band_mask, :]
    ecg_reconstructed = Tx_band.real.sum(axis=0)
    return np.real(ecg_reconstructed)


if HAS_SSQ:
    SSQ_DUR = 10.0
    ssq_start = int(VIEW_START_S * biopac["fs"])
    ssq_end   = int((VIEW_START_S + SSQ_DUR) * biopac["fs"])
    ssq_sig   = biopac["ecg_raw"][ssq_start:ssq_end].astype(float)
    ssq_t     = biopac["time_s"][ssq_start:ssq_end]
    ssq_fs    = biopac["fs"]

    print(f"Computing SSQ-CWT on {SSQ_DUR}s segment ({len(ssq_sig)} samples)...")
    Tx, Wx, ssq_freqs, scales, *_ = ssq_cwt(ssq_sig, fs=ssq_fs)
    ssq_mag = np.abs(Tx)

    # -- Plot 1: SSQ-CWT heatmap --
    freq_mask = (ssq_freqs >= 0.5) & (ssq_freqs <= 100)
    freqs_vis = ssq_freqs[freq_mask]
    mag_vis = ssq_mag[freq_mask, :]
    vmax = np.percentile(mag_vis, 99)

    if HAS_PLOTLY:
        fig_ssq = go.Figure(data=go.Heatmap(
            z=mag_vis,
            x=ssq_t,
            y=freqs_vis,
            colorscale="Viridis",
            zmin=0, zmax=vmax,
            colorbar=dict(title="Magnitude"),
        ))
        fig_ssq.add_hline(y=5, line_dash="dash", line_color="red",
                          annotation_text="5 Hz")
        fig_ssq.add_hline(y=40, line_dash="dash", line_color="red",
                          annotation_text="40 Hz")
        fig_ssq.update_layout(
            title="SSQ-CWT Time-Frequency Representation (BIOPAC)",
            xaxis_title="Time (s)",
            yaxis_title="Frequency (Hz)",
            height=500, width=900,
        )
        fig_ssq.show()
    else:
        fig, ax = plt.subplots(figsize=(14, 5))
        im = ax.pcolormesh(ssq_t, freqs_vis, mag_vis, shading="auto",
                           cmap="viridis", vmin=0, vmax=vmax)
        ax.axhline(5, color="red", linestyle="--", linewidth=1, label="5 Hz")
        ax.axhline(40, color="red", linestyle="--", linewidth=1, label="40 Hz")
        ax.set_title("SSQ-CWT Time-Frequency Representation (BIOPAC)", fontweight="bold")
        ax.set_xlabel("Time (s)")
        ax.set_ylabel("Frequency (Hz)")
        ax.legend(loc="upper right")
        plt.colorbar(im, ax=ax, label="Magnitude")
        fig.tight_layout()
        fig.savefig(os.path.join(OUTPUT_DIR, "ssq_cwt_heatmap.png"), dpi=150, bbox_inches="tight")
        plt.show()

    # -- Plot 2: Comparison overlay --
    ssq_recon = ssq_ecg_band_reconstruct(ssq_sig, ssq_fs, fmin=5, fmax=40)
    ssq_bp    = bandpass_filter(ssq_sig, ssq_fs, 0.5, 40.0)

    fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
    axes[0].plot(ssq_t, ssq_sig, color="steelblue", linewidth=0.5)
    axes[0].set_title("Raw ECG", fontweight="bold")
    axes[0].set_ylabel("Amplitude (mV)")

    axes[1].plot(ssq_t, ssq_bp, color="teal", linewidth=0.5)
    axes[1].set_title("Butterworth Bandpass (0.5-40 Hz)", fontweight="bold")
    axes[1].set_ylabel("Amplitude (mV)")

    axes[2].plot(ssq_t, ssq_recon, color="darkviolet", linewidth=0.5)
    axes[2].set_title("SSQ-CWT Reconstructed (5-40 Hz band)", fontweight="bold")
    axes[2].set_ylabel("Amplitude (mV)")
    axes[2].set_xlabel("Time (s)")

    for ax in axes:
        ax.grid(True, alpha=0.3)

    fig.suptitle("Bandpass vs SSQ-CWT Reconstruction", fontsize=13, fontweight="bold", y=1.01)
    fig.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, "ssq_vs_bandpass.png"), dpi=150, bbox_inches="tight")
    plt.show()


    # ---- Belt SSQ-CWT ----
    bl_ssq_start = int(VIEW_START_S * belt["fs"])
    bl_ssq_end   = int((VIEW_START_S + SSQ_DUR) * belt["fs"])
    if bl_ssq_end <= len(belt["ecg"]):
        bl_ssq_sig = belt["ecg"][bl_ssq_start:bl_ssq_end].astype(float)
        bl_ssq_t   = belt["time_s"][bl_ssq_start:bl_ssq_end]
        bl_ssq_fs  = belt["fs"]

        print(f"Computing SSQ-CWT on Belt {SSQ_DUR}s segment ({len(bl_ssq_sig)} samples)...")
        bl_Tx, bl_Wx, bl_ssq_freqs, bl_scales, *_ = ssq_cwt(bl_ssq_sig, fs=bl_ssq_fs)
        bl_ssq_mag = np.abs(bl_Tx)

        bl_freq_mask = (bl_ssq_freqs >= 0.5) & (bl_ssq_freqs <= bl_ssq_fs / 2)
        bl_freqs_vis = bl_ssq_freqs[bl_freq_mask]
        bl_mag_vis = bl_ssq_mag[bl_freq_mask, :]
        bl_vmax = np.percentile(bl_mag_vis, 99)

        if HAS_PLOTLY:
            fig_bl_ssq = go.Figure(data=go.Heatmap(
                z=bl_mag_vis, x=bl_ssq_t, y=bl_freqs_vis,
                colorscale="Viridis", zmin=0, zmax=bl_vmax,
                colorbar=dict(title="Magnitude"),
            ))
            fig_bl_ssq.add_hline(y=5, line_dash="dash", line_color="red",
                                 annotation_text="5 Hz")
            fig_bl_ssq.add_hline(y=40, line_dash="dash", line_color="red",
                                 annotation_text="40 Hz")
            fig_bl_ssq.update_layout(
                title="SSQ-CWT Time-Frequency Representation (Baby Belt)",
                xaxis_title="Time (s)", yaxis_title="Frequency (Hz)",
                height=500, width=900,
            )
            fig_bl_ssq.show()
        else:
            fig, ax = plt.subplots(figsize=(14, 5))
            im = ax.pcolormesh(bl_ssq_t, bl_freqs_vis, bl_mag_vis,
                               shading="auto", cmap="viridis", vmin=0, vmax=bl_vmax)
            ax.axhline(5, color="red", linestyle="--", linewidth=1, label="5 Hz")
            ax.axhline(40, color="red", linestyle="--", linewidth=1, label="40 Hz")
            ax.set_title("SSQ-CWT Time-Frequency Representation (Baby Belt)", fontweight="bold")
            ax.set_xlabel("Time (s)")
            ax.set_ylabel("Frequency (Hz)")
            ax.legend(loc="upper right")
            plt.colorbar(im, ax=ax, label="Magnitude")
            fig.tight_layout()
            fig.savefig(os.path.join(OUTPUT_DIR, "ssq_cwt_heatmap_belt.png"), dpi=150, bbox_inches="tight")
            plt.show()
    else:
        print("Belt segment too short for SSQ-CWT demo.")
    print("SSQ-CWT demo complete (BIOPAC + Belt).")
else:
    print("Skipping SSQ-CWT demo (ssqueezepy not installed).")

## 9. Wavelet Transform Visualisation (PyWavelets CWT)

The **Continuous Wavelet Transform (CWT)** gives a time-scale (time-frequency)
representation of the signal. Unlike the FFT, it preserves temporal localisation --
you can see *when* each frequency component occurs.

We use `pywt.cwt()` with a complex Morlet wavelet and display the result as an
interactive heatmap. The **colormap is rescaled** based on the displayed frequency
window to prevent low-frequency energy from oversaturating the QRS band.

> Adjust `CWT_FMIN` and `CWT_FMAX` below to explore different frequency bands.

In [ ]:
# =====================================================================
# CWT CONFIGURATION
# =====================================================================
CWT_FMIN  = 1.0    # Min frequency to display (Hz)
CWT_FMAX  = 50.0   # Max frequency to display (Hz)
CWT_DUR_S = 10.0   # Duration of segment to analyse (seconds)
# =====================================================================

if HAS_PYW:
    cwt_start = int(VIEW_START_S * biopac["fs"])
    cwt_end   = int((VIEW_START_S + CWT_DUR_S) * biopac["fs"])
    cwt_sig   = biopac["ecg_raw"][cwt_start:cwt_end].astype(float)
    cwt_t     = biopac["time_s"][cwt_start:cwt_end]
    cwt_fs    = biopac["fs"]

    freqs = np.linspace(CWT_FMIN, CWT_FMAX, 200)
    scales = pywt.central_frequency("cmor1.5-1.0") * cwt_fs / freqs

    print(f"Computing CWT ({len(scales)} scales, {len(cwt_sig)} samples)...")
    coeffs, _ = pywt.cwt(cwt_sig, scales, "cmor1.5-1.0",
                          sampling_period=1.0 / cwt_fs)
    power = np.abs(coeffs)
    vmax_cwt = np.percentile(power, 99)

    if HAS_PLOTLY:
        fig_cwt = go.Figure(data=go.Heatmap(
            z=power,
            x=cwt_t,
            y=freqs,
            colorscale="Magma",
            zmin=0, zmax=vmax_cwt,
            colorbar=dict(title="Magnitude"),
        ))
        fig_cwt.add_hline(y=5, line_dash="dash", line_color="cyan",
                          annotation_text="5 Hz (QRS low)")
        fig_cwt.add_hline(y=40, line_dash="dash", line_color="cyan",
                          annotation_text="40 Hz (QRS high)")
        fig_cwt.update_layout(
            title=f"CWT Scalogram -- Morlet Wavelet ({CWT_FMIN}-{CWT_FMAX} Hz)",
            xaxis_title="Time (s)",
            yaxis_title="Frequency (Hz)",
            height=500, width=900,
        )
        fig_cwt.show()
    else:
        fig, ax = plt.subplots(figsize=(14, 5))
        im = ax.pcolormesh(cwt_t, freqs, power, shading="auto",
                           cmap="magma", vmin=0, vmax=vmax_cwt)
        ax.axhline(5, color="cyan", linestyle="--", linewidth=1, label="5 Hz")
        ax.axhline(40, color="cyan", linestyle="--", linewidth=1, label="40 Hz")
        ax.set_title(f"CWT Scalogram -- Morlet ({CWT_FMIN}-{CWT_FMAX} Hz)", fontweight="bold")
        ax.set_xlabel("Time (s)")
        ax.set_ylabel("Frequency (Hz)")
        ax.legend(loc="upper right")
        plt.colorbar(im, ax=ax, label="Magnitude")
        fig.tight_layout()
        fig.savefig(os.path.join(OUTPUT_DIR, "cwt_scalogram.png"), dpi=150, bbox_inches="tight")
        plt.show()

    # QRS band energy overlay
    qrs_mask = (freqs >= 5) & (freqs <= 40)
    qrs_power_sum = np.sum(power[qrs_mask, :], axis=0)
    qrs_power_norm = qrs_power_sum / qrs_power_sum.max()

    fig, ax1 = plt.subplots(figsize=(14, 4))
    ax1.plot(cwt_t, cwt_sig, color="steelblue", linewidth=0.5, label="Raw ECG")
    ax1.set_ylabel("ECG Amplitude (mV)", color="steelblue")
    ax1.set_xlabel("Time (s)")

    ax2 = ax1.twinx()
    ax2.fill_between(cwt_t, 0, qrs_power_norm, color="red", alpha=0.2, label="QRS band energy")
    ax2.set_ylabel("Normalised QRS energy", color="red")

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right")
    ax1.set_title("Raw ECG with CWT QRS-Band Energy Overlay", fontweight="bold")
    ax1.grid(True, alpha=0.3)

    fig.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, "cwt_qrs_energy_overlay.png"), dpi=150, bbox_inches="tight")
    plt.show()


    # ---- Belt CWT Scalogram ----
    bl_cwt_start = int(VIEW_START_S * belt["fs"])
    bl_cwt_end   = int((VIEW_START_S + CWT_DUR_S) * belt["fs"])
    if bl_cwt_end <= len(belt["ecg"]):
        bl_cwt_sig = belt["ecg"][bl_cwt_start:bl_cwt_end].astype(float)
        bl_cwt_t   = belt["time_s"][bl_cwt_start:bl_cwt_end]
        bl_cwt_fs  = belt["fs"]

        bl_fmax = min(CWT_FMAX, bl_cwt_fs / 2.0 - 1)
        bl_freqs = np.linspace(CWT_FMIN, bl_fmax, 200)
        bl_scales_cwt = pywt.central_frequency("cmor1.5-1.0") * bl_cwt_fs / bl_freqs

        print(f"Computing Belt CWT ({len(bl_scales_cwt)} scales, {len(bl_cwt_sig)} samples)...")
        bl_coeffs, _ = pywt.cwt(bl_cwt_sig, bl_scales_cwt, "cmor1.5-1.0",
                                 sampling_period=1.0 / bl_cwt_fs)
        bl_power = np.abs(bl_coeffs)
        bl_vmax_cwt = np.percentile(bl_power, 99)

        if HAS_PLOTLY:
            fig_bl_cwt = go.Figure(data=go.Heatmap(
                z=bl_power, x=bl_cwt_t, y=bl_freqs,
                colorscale="Magma", zmin=0, zmax=bl_vmax_cwt,
                colorbar=dict(title="Magnitude"),
            ))
            fig_bl_cwt.add_hline(y=5, line_dash="dash", line_color="cyan",
                                 annotation_text="5 Hz (QRS low)")
            fig_bl_cwt.add_hline(y=40, line_dash="dash", line_color="cyan",
                                 annotation_text="40 Hz (QRS high)")
            fig_bl_cwt.update_layout(
                title=f"Belt CWT Scalogram -- Morlet ({CWT_FMIN}-{bl_fmax:.0f} Hz)",
                xaxis_title="Time (s)", yaxis_title="Frequency (Hz)",
                height=500, width=900,
            )
            fig_bl_cwt.show()
        else:
            fig, ax = plt.subplots(figsize=(14, 5))
            im = ax.pcolormesh(bl_cwt_t, bl_freqs, bl_power,
                               shading="auto", cmap="magma", vmin=0, vmax=bl_vmax_cwt)
            ax.axhline(5, color="cyan", linestyle="--", linewidth=1, label="5 Hz")
            ax.axhline(40, color="cyan", linestyle="--", linewidth=1, label="40 Hz")
            ax.set_title(f"Belt CWT Scalogram -- Morlet ({CWT_FMIN}-{bl_fmax:.0f} Hz)", fontweight="bold")
            ax.set_xlabel("Time (s)")
            ax.set_ylabel("Frequency (Hz)")
            ax.legend(loc="upper right")
            plt.colorbar(im, ax=ax, label="Magnitude")
            fig.tight_layout()
            fig.savefig(os.path.join(OUTPUT_DIR, "cwt_scalogram_belt.png"), dpi=150, bbox_inches="tight")
            plt.show()

        bl_qrs_mask = (bl_freqs >= 5) & (bl_freqs <= 40)
        bl_qrs_power_sum = np.sum(bl_power[bl_qrs_mask, :], axis=0)
        bl_qrs_power_norm = bl_qrs_power_sum / bl_qrs_power_sum.max()

        fig, ax1 = plt.subplots(figsize=(14, 4))
        ax1.plot(bl_cwt_t, bl_cwt_sig, color="darkorange", linewidth=0.5, label="Raw Belt ECG")
        ax1.set_ylabel("ECG Amplitude (ADC)", color="darkorange")
        ax1.set_xlabel("Time (s)")

        ax2 = ax1.twinx()
        ax2.fill_between(bl_cwt_t, 0, bl_qrs_power_norm, color="red", alpha=0.2, label="QRS band energy")
        ax2.set_ylabel("Normalised QRS energy", color="red")

        lines1, labels1 = ax1.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right")
        ax1.set_title("Belt ECG with CWT QRS-Band Energy Overlay", fontweight="bold")
        ax1.grid(True, alpha=0.3)

        fig.tight_layout()
        fig.savefig(os.path.join(OUTPUT_DIR, "cwt_qrs_energy_overlay_belt.png"), dpi=150, bbox_inches="tight")
        plt.show()
    else:
        print("Belt segment too short for CWT.")
    print("CWT visualisation complete (BIOPAC + Belt).")
else:
    print("Skipping CWT section (PyWavelets not installed).")

## 10. BIOPAC vs Belt -- Full Pipeline Comparison

We now visualise both signals side-by-side through three processing stages:

| Stage | Processing applied |
|---|---|
| **Raw** | No processing |
| **Bandpass** | Butterworth bandpass 0.5-40 Hz |
| **Full pipeline** | Notch (60 Hz) -> Bandpass (0.5-40 Hz) -> Baseline removal |

This makes the quality difference between gelled and textile electrodes -- and
the degree to which our pipeline narrows that gap -- immediately visible.

In [ ]:
# Full pipeline for Belt (re-derive for completeness)
bl_notched  = notch_filter(belt["ecg"], belt["fs"], freq=60.0, quality=30.0)
bl_bp_full  = bandpass_filter(bl_notched, belt["fs"], 0.5, 40.0)
bl_final, _ = remove_baseline(bl_bp_full, belt["fs"])

bp_final = bp_step3

# --- 6-panel vertical figure ---
fig, axes = plt.subplots(3, 2, figsize=(16, 12), sharex="col")

t_bp_w = biopac["time_s"]
t_bl_w = belt["time_s"]
mask_bp_w = (t_bp_w >= VIEW_START_S) & (t_bp_w < VIEW_END_S)
mask_bl_w = (t_bl_w >= VIEW_START_S) & (t_bl_w < VIEW_END_S)

# Row 0: Raw
axes[0, 0].plot(t_bp_w[mask_bp_w], biopac["ecg_raw"][mask_bp_w],
                color="steelblue", linewidth=0.5)
axes[0, 0].set_title("BIOPAC -- Raw", fontweight="bold")
axes[0, 0].set_ylabel("mV")

axes[0, 1].plot(t_bl_w[mask_bl_w], belt["ecg"][mask_bl_w],
                color="darkorange", linewidth=0.5)
axes[0, 1].set_title("Belt -- Raw", fontweight="bold")
axes[0, 1].set_ylabel("ADC counts")

# Row 1: Bandpass only
axes[1, 0].plot(t_bp_w[mask_bp_w], bp_ecg_bp[mask_bp_w],
                color="teal", linewidth=0.5)
axes[1, 0].set_title("BIOPAC -- Bandpass (0.5-40 Hz)", fontweight="bold")
axes[1, 0].set_ylabel("mV")

axes[1, 1].plot(t_bl_w[mask_bl_w], bl_ecg_bp[mask_bl_w],
                color="darkred", linewidth=0.5)
axes[1, 1].set_title("Belt -- Bandpass (0.5-40 Hz)", fontweight="bold")
axes[1, 1].set_ylabel("ADC counts")

# Row 2: Full pipeline
axes[2, 0].plot(t_bp_w[mask_bp_w], bp_final[mask_bp_w],
                color="darkgreen", linewidth=0.5)
axes[2, 0].set_title("BIOPAC -- Full Pipeline", fontweight="bold")
axes[2, 0].set_ylabel("mV")
axes[2, 0].set_xlabel("Time (s)")

axes[2, 1].plot(t_bl_w[mask_bl_w], bl_final[mask_bl_w],
                color="darkviolet", linewidth=0.5)
axes[2, 1].set_title("Belt -- Full Pipeline", fontweight="bold")
axes[2, 1].set_ylabel("ADC counts")
axes[2, 1].set_xlabel("Time (s)")

for ax_row in axes:
    for ax in ax_row:
        ax.grid(True, alpha=0.3)

fig.suptitle("BIOPAC vs Baby Belt -- Preprocessing Pipeline Comparison",
             fontsize=14, fontweight="bold", y=1.01)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "biopac_vs_belt_pipeline.png"), dpi=150, bbox_inches="tight")
plt.show()

print("Final comparison figure saved.")

## Summary & Next Steps

### What we covered

| Step | Technique | Key function |
|---|---|---|
| **Bandpass** | Butterworth, zero-phase (`sosfiltfilt`) | `bandpass_filter()` |
| **Notch** | IIR notch via `iirnotch` + `tf2sos` | `notch_filter()` |
| **Baseline removal** | Double median filter | `remove_baseline()` |
| **Synthetic noise** | Baseline wander, Gaussian, PLI, EMG generators | `add_synthetic_*()` |
| **Time-frequency** | SSQ-CWT (ssqueezepy) and CWT (PyWavelets) | `ssq_ecg_band_reconstruct()` |

### Key takeaways

- Textile ECG (Baby Belt) exhibits substantially more baseline wander and motion
  artifact than gelled ECG (BIOPAC)
- A simple three-stage pipeline (notch -> bandpass -> baseline removal) dramatically
  improves signal quality for both modalities
- Synchrosqueezed CWT offers sharper frequency localisation than standard CWT,
  useful for separating overlapping spectral components
- Synthetic noise injection helps build intuition about filter performance and is
  essential for training robust neural-network detectors (see NB05/NB06)

### Next notebooks

- **NB03** -- R-Peak Detection: algorithmic and NeuroKit2-based QRS detection on preprocessed signals
- **NB04** -- Heart Rate Variability: time-domain and frequency-domain HRV metrics
- **NB05** -- RPNet Architecture: deep learning R-peak detection on noisy textile ECG
- **NB06** -- Evaluation & Benchmarking: comparing detector performance across noise levels

---
*All outputs saved to `outputs/NB02/`.*